In [ ]:
from sim_stim import *
from adaptive_latents import datasets
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

In [ ]:
rng = np.random.default_rng()
d = datasets.Zong22Dataset()
data = d.neural_data

stim_magnitude = 10
design_method = 'optimized identity u_to_s'
exit_time = 150
stim_rate = None
smoothing_tau = 1
centerer_init_size = 8 * 25
initial_nostim_period = 30
regular_stim_iter = cycle([1 / 10, 1 / 3])
stim_timing_method = 'regular'

val = dict(attempt_correction=True, heed_stimuli=True, exit_time=exit_time,
           stim_magnitude=stim_magnitude, design_method=design_method, stim_rate=stim_rate,
           smoothing_tau=smoothing_tau, centerer_init_size=centerer_init_size,
           initial_nostim_period=initial_nostim_period,
           regular_stim_iter =regular_stim_iter, stim_timing_method=stim_timing_method)

sr, stim_designer, log = make_sr(input_array=data, rng=rng.spawn(1)[0], show_tqdm=True, **val)


In [ ]:
i= 0

fig, axs = plt.subplots(ncols=2, figsize=(10,4), sharex=False, sharey=False, layout='constrained')

latents = log['latents'].slice_by_time(slice(30,None))
axs[0].plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')
stim_s = log['stim_intended_samples'].t - latents.dt

l = 100
r = 200
ax_n = 0
center_t = log['stim_intended_samples'].t[i]
latents = log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = axs[ax_n].plot(latents[:, 0], latents[:, 1])
stim_s = log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
axs[ax_n].plot(latents_s[:, 0], latents_s[:, 1], '.', color='r')

for arrow_index in [17, 50]:
    axs[0].annotate('',
                    xytext=(latents[arrow_index, 0], latents[arrow_index, 1]),
                    xy=(latents[arrow_index+1, 0], latents[arrow_index+1, 1]),
                    arrowprops=dict(arrowstyle="simple", color='C0'),
                    size=11
                    )


u = stim_designer.log[i]['u']
idx = np.argsort(np.abs(u))[::-1]
print(np.linalg.norm(u,ord=0))

high_d = log['high_d_with_stim'].slice_by_time(slice(center_t-l,center_t+r))
axs[1].plot(high_d.t, high_d[:,idx[:int(np.linalg.norm(u,ord=0))]]);
for stim_t in stim_s:
    axs[1].axvline(stim_t, color='r')
